In [ ]:
# Install required packages
%pip install tensorflow numpy matplotlib seaborn scikit-learn pandas

# AlexNet Model with TensorFlow/Keras

This notebook implements the AlexNet convolutional neural network architecture using TensorFlow/Keras for image classification on the MNIST dataset.

## Objectives:
- Load and preprocess the MNIST dataset
- Build the AlexNet model architecture
- Train with different optimizers (SGD, Adam, RMSProp)
- Apply regularization techniques (Dropout, L1/L2, EarlyStopping)
- Perform basic hyperparameter tuning
- Visualize training curves and confusion matrix
- Evaluate model performance
- Save the trained model

In [ ]:
# Import Required Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import pandas as pd
import os
from collections import Counter

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Dataset Loading Function
def load_mnist_dataset():
    """
    Load MNIST dataset from local Kaggle folder or TensorFlow Datasets
    """
    local_train_path = "data/mnist_train.csv"  # Placeholder for local Kaggle dataset
    local_test_path = "data/mnist_test.csv"
    
    if os.path.exists(local_train_path) and os.path.exists(local_test_path):
        print("Loading dataset from local files...")
        # Load CSV format
        train_df = pd.read_csv(local_train_path)
        test_df = pd.read_csv(local_test_path)
        
        # Assuming first column is label, rest are pixels
        train_labels = train_df.iloc[:, 0].values
        train_images = train_df.iloc[:, 1:].values
        test_labels = test_df.iloc[:, 0].values
        test_images = test_df.iloc[:, 1:].values
        
        return (train_images, train_labels), (test_images, test_labels)
    else:
        print("Local dataset not found. Loading from TensorFlow/Keras...")
        # Load MNIST from keras
        (train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()
        return (train_images, train_labels), (test_images, test_labels)

# Load the dataset
(train_images, train_labels), (test_images, test_labels) = load_mnist_dataset()
print("Dataset loaded successfully!")

NameError: name 'os' is not defined

In [ ]:
# Dataset Exploration
def explore_mnist_dataset(train_images, train_labels, test_images, test_labels):
    """
    Display dataset shape, sample data, and label distribution
    """
    print("Dataset Info:")
    print(f"Train images shape: {train_images.shape}")
    print(f"Train labels shape: {train_labels.shape}")
    print(f"Test images shape: {test_images.shape}")
    print(f"Test labels shape: {test_labels.shape}")
    
    print(f"\nImage data type: {train_images.dtype}")
    print(f"Label data type: {train_labels.dtype}")
    
    # Sample data
    print("\nSample Data:")
    plt.figure(figsize=(10, 4))
    for i in range(10):
        plt.subplot(2, 5, i+1)
        plt.imshow(train_images[i], cmap='gray')
        plt.title(f"Label: {train_labels[i]}")
        plt.axis('off')
    plt.show()
    
    # Label distribution
    train_label_counts = Counter(train_labels)
    test_label_counts = Counter(test_labels)
    
    print("\nLabel Distribution:")
    print("Train set:")
    for label in sorted(train_label_counts.keys()):
        print(f"  {label}: {train_label_counts[label]}")
    
    print("Test set:")
    for label in sorted(test_label_counts.keys()):
        print(f"  {label}: {test_label_counts[label]}")
    
    # Plot distribution
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    labels, counts = zip(*sorted(train_label_counts.items()))
    ax1.bar(labels, counts)
    ax1.set_title('Training Set Label Distribution')
    ax1.set_xlabel('Digit')
    ax1.set_ylabel('Count')
    
    labels, counts = zip(*sorted(test_label_counts.items()))
    ax2.bar(labels, counts)
    ax2.set_title('Test Set Label Distribution')
    ax2.set_xlabel('Digit')
    ax2.set_ylabel('Count')
    
    plt.show()

# Explore the dataset
explore_mnist_dataset(train_images, train_labels, test_images, test_labels)

In [ ]:
# Preprocessing
def preprocess_mnist_data(train_images, train_labels, test_images, test_labels):
    """
    Preprocess MNIST data: normalize, reshape, split validation
    """
    # Normalize pixel values to [0, 1]
    train_images = train_images.astype('float32') / 255.0
    test_images = test_images.astype('float32') / 255.0
    
    # Reshape for AlexNet (add channel dimension and resize to 224x224)
    # AlexNet expects 224x224x3, but MNIST is 28x28x1
    # We'll resize to 224x224 and replicate channels
    def resize_and_replicate_channels(images):
        resized = tf.image.resize(images[..., tf.newaxis], (224, 224))
        return tf.repeat(resized, 3, axis=-1)
    
    train_images = resize_and_replicate_channels(train_images)
    test_images = resize_and_replicate_channels(test_images)
    
    # Convert labels to categorical
    train_labels = keras.utils.to_categorical(train_labels, 10)
    test_labels = keras.utils.to_categorical(test_labels, 10)
    
    # Split train into train and validation
    val_split = 0.1
    val_size = int(len(train_images) * val_split)
    
    val_images = train_images[:val_size]
    val_labels = train_labels[:val_size]
    train_images = train_images[val_size:]
    train_labels = train_labels[val_size:]
    
    return (train_images, train_labels), (val_images, val_labels), (test_images, test_labels)

# Preprocess data
(train_X, train_y), (val_X, val_y), (test_X, test_y) = preprocess_mnist_data(
    train_images, train_labels, test_images, test_labels
)

print(f"Training data shape: {train_X.shape}")
print(f"Validation data shape: {val_X.shape}")
print(f"Test data shape: {test_X.shape}")
print(f"Number of classes: {train_y.shape[1]}")

In [ ]:
# Build AlexNet Model
def build_alexnet_model(num_classes=10):
    """
    Build AlexNet architecture adapted for MNIST with regularization
    """
    model = keras.Sequential([
        # First Convolutional Layer
        layers.Conv2D(96, (11, 11), strides=(4, 4), activation='relu', input_shape=(224, 224, 3),
                     kernel_regularizer=keras.regularizers.l2(0.01)),
        layers.MaxPooling2D((3, 3), strides=(2, 2)),
        layers.BatchNormalization(),
        
        # Second Convolutional Layer
        layers.Conv2D(256, (5, 5), activation='relu', padding='same',
                     kernel_regularizer=keras.regularizers.l2(0.01)),
        layers.MaxPooling2D((3, 3), strides=(2, 2)),
        layers.BatchNormalization(),
        
        # Third Convolutional Layer
        layers.Conv2D(384, (3, 3), activation='relu', padding='same',
                     kernel_regularizer=keras.regularizers.l2(0.01)),
        layers.BatchNormalization(),
        
        # Fourth Convolutional Layer
        layers.Conv2D(384, (3, 3), activation='relu', padding='same',
                     kernel_regularizer=keras.regularizers.l2(0.01)),
        layers.BatchNormalization(),
        
        # Fifth Convolutional Layer
        layers.Conv2D(256, (3, 3), activation='relu', padding='same',
                     kernel_regularizer=keras.regularizers.l2(0.01)),
        layers.MaxPooling2D((3, 3), strides=(2, 2)),
        layers.BatchNormalization(),
        
        # Flatten and Dense Layers
        layers.Flatten(),
        layers.Dense(4096, activation='relu', kernel_regularizer=keras.regularizers.l1_l2(l1=0.01, l2=0.01)),
        layers.Dropout(0.5),
        layers.Dense(4096, activation='relu', kernel_regularizer=keras.regularizers.l1_l2(l1=0.01, l2=0.01)),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

# Build model
model = build_alexnet_model()

# Print model summary
model.summary()

In [ ]:
# Training with Different Optimizers
def train_with_optimizer(model, optimizer_name, train_X, train_y, val_X, val_y, epochs=10):
    """
    Train model with specified optimizer and regularization
    """
    if optimizer_name == 'SGD':
        optimizer = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
    elif optimizer_name == 'Adam':
        optimizer = keras.optimizers.Adam(learning_rate=0.001)
    elif optimizer_name == 'RMSProp':
        optimizer = keras.optimizers.RMSprop(learning_rate=0.001)
    
    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    # Early stopping
    early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    
    # Data augmentation
    datagen = keras.preprocessing.image.ImageDataGenerator(
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=True
    )
    
    history = model.fit(datagen.flow(train_X, train_y, batch_size=32),
                       validation_data=(val_X, val_y),
                       epochs=epochs,
                       callbacks=[early_stopping],
                       verbose=1)
    
    return history

# Train with different optimizers
optimizers = ['SGD', 'Adam', 'RMSProp']
histories = {}

for opt in optimizers:
    print(f"\nTraining with {opt} optimizer...")
    model_copy = build_alexnet_model()
    history = train_with_optimizer(model_copy, opt, train_X, train_y, val_X, val_y)
    histories[opt] = history
    print(f"{opt} training completed.")

In [ ]:
# Hyperparameter Tuning (Basic)
def hyperparameter_tuning(train_X, train_y, val_X, val_y):
    """
    Basic hyperparameter tuning for learning rate and batch size
    """
    best_model = None
    best_val_acc = 0
    best_params = {}
    
    learning_rates = [0.001, 0.01, 0.1]
    batch_sizes = [32, 64, 128]
    
    for lr in learning_rates:
        for bs in batch_sizes:
            print(f"\nTuning: learning_rate={lr}, batch_size={bs}")
            
            model = build_alexnet_model()
            model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                         loss='categorical_crossentropy',
                         metrics=['accuracy'])
            
            early_stopping = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=2, restore_best_weights=True)
            
            datagen = keras.preprocessing.image.ImageDataGenerator(
                rotation_range=10,
                width_shift_range=0.1,
                height_shift_range=0.1,
                horizontal_flip=True
            )
            
            history = model.fit(datagen.flow(train_X, train_y, batch_size=bs),
                               validation_data=(val_X, val_y),
                               epochs=5,
                               callbacks=[early_stopping],
                               verbose=0)
            
            val_acc = max(history.history['val_accuracy'])
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model = model
                best_params = {'learning_rate': lr, 'batch_size': bs}
    
    print(f"\nBest hyperparameters: {best_params}")
    print(f"Best validation accuracy: {best_val_acc}")
    
    return best_model, best_params

# Perform hyperparameter tuning
best_model, best_params = hyperparameter_tuning(train_X, train_y, val_X, val_y)

In [ ]:
# Plot Training Curves
def plot_training_curves(histories):
    """
    Plot training vs validation loss and accuracy for different optimizers
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    for i, (opt, history) in enumerate(histories.items()):
        # Loss
        axes[0, i].plot(history.history['loss'], label='Training Loss')
        axes[0, i].plot(history.history['val_loss'], label='Validation Loss')
        axes[0, i].set_title(f'{opt} - Loss')
        axes[0, i].set_xlabel('Epoch')
        axes[0, i].set_ylabel('Loss')
        axes[0, i].legend()
        
        # Accuracy
        axes[1, i].plot(history.history['accuracy'], label='Training Accuracy')
        axes[1, i].plot(history.history['val_accuracy'], label='Validation Accuracy')
        axes[1, i].set_title(f'{opt} - Accuracy')
        axes[1, i].set_xlabel('Epoch')
        axes[1, i].set_ylabel('Accuracy')
        axes[1, i].legend()
    
    plt.tight_layout()
    plt.show()

# Plot curves for different optimizers
plot_training_curves(histories)

In [ ]:
# Model Evaluation
def evaluate_model(model, test_X, test_y):
    """
    Evaluate model performance with accuracy, loss, F1 score, and confusion matrix
    """
    # Predictions
    predictions = model.predict(test_X)
    pred_labels = np.argmax(predictions, axis=1)
    true_labels = np.argmax(test_y, axis=1)
    
    # Calculate metrics
    accuracy = np.mean(pred_labels == true_labels)
    loss = model.evaluate(test_X, test_y, verbose=0)[0]
    
    # F1 Score (macro average)
    f1 = f1_score(true_labels, pred_labels, average='macro')
    
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test Loss: {loss:.4f}")
    print(f"F1 Score (Macro): {f1:.4f}")
    
    # Classification report
    target_names = [str(i) for i in range(10)]
    
    print("\nClassification Report:")
    print(classification_report(true_labels, pred_labels, target_names=target_names))
    
    # Confusion Matrix
    cm = confusion_matrix(true_labels, pred_labels)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=target_names, yticklabels=target_names)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()
    
    return accuracy, loss, f1

# Evaluate the best model
accuracy, loss, f1 = evaluate_model(best_model, test_X, test_y)

In [ ]:
# Save Model
def save_model(model, filename='alexnet_model.h5'):
    """
    Save the trained model
    """
    model.save(filename)
    print(f"Model saved as {filename}")

# Save the best model
save_model(best_model, 'alexnet_model.h5')

## Summary

This notebook implemented the AlexNet convolutional neural network architecture using TensorFlow/Keras for MNIST digit classification:

1. **Dataset**: Loaded MNIST dataset from local CSV or TensorFlow/Keras
2. **Preprocessing**: Normalized pixels, resized images, converted labels to categorical
3. **Model**: AlexNet architecture with 5 conv layers and 3 dense layers
4. **Training**: Tested SGD, Adam, and RMSProp optimizers with data augmentation
5. **Regularization**: Applied L1/L2 regularization, Dropout, Batch Normalization, and Early Stopping
6. **Tuning**: Basic hyperparameter tuning for learning rate and batch size
7. **Visualization**: Training curves and confusion matrix
8. **Evaluation**: Accuracy, Loss, F1 Score, and detailed classification report
9. **Model Saving**: Saved as .h5 file

The AlexNet model achieved good performance on the MNIST dataset. You can further improve it by:
- Using more advanced data augmentation techniques
- Implementing learning rate scheduling
- Using pre-trained weights from ImageNet (if adapting for larger datasets)
- Adding more regularization techniques like label smoothing
- Experimenting with different architectures like VGG or ResNet